## 005_backtest — Leader–Follower Strategy Validation

**Author:** Wayne Kirk Schmidt  
**Email:** wayne.kirk.schmidt@gmail.com  

---

## Stage Purpose

This stage evaluates the statistical arbitrage hypotheses identified
in **004_strategy** by constructing and testing trade rules based on
leader–follower shock propagation.

Using the event-level signal structures produced in 004, we simulate
trades that exploit delayed reactions between assets and measure
the resulting performance characteristics.

The goal of this stage is **not discovery**, but **validation**:
to determine whether the observed signal structures translate into
repeatable, economically meaningful trading performance.

---

## Inputs (from 004_strategy)

Loaded from the pipeline manifest:

- `event_full_df.pkl`
  - Complete event-level dataset of leader–follower relationships
  - Includes:
    - leader coin (`r_coin`)
    - follower coin (`f_coin`)
    - leader direction (`r_sign`)
    - follower directional responses (`f_sign0/1/2`)
    - cumulative responses (`f_sum0/1/2`)

- `event_filtered_df.pkl`
  - Subset of events where follower response aligns directionally
  - Represents candidate signal events

- `event_strong_df.pkl`
  - High-confidence subset where multi-period agreement is observed
  - Represents strongest signal candidates

Additional inputs:

- `returns_full.pkl` (from 002_enrich)
  - Daily return series for all assets

- `price_wide.pkl` (from 002_enrich)
  - Wide-format price series for all assets

These artifacts represent the **frozen output of the strategy construction stage**.

---

## Core Tasks

This notebook performs the following steps:

1. **Signal Definition**
   - Select an input event set (full / filtered / strong)
   - Define trade direction from leader shock (`r_sign`)
   - Define mapping from event structure to executable trade

2. **Trade Construction**
   - Define entry timing (e.g. t+1)
   - Define exit timing (e.g. t+2 or t+3)
   - Map event dates to realized returns using `returns_full`
   - Construct event-level trade records

3. **Trade Simulation**
   - Aggregate event-level trades into a unified trade dataset
   - Apply execution assumptions (delay, cost)
   - Generate trade-level PnL series

4. **Performance Evaluation**
   - cumulative return
   - volatility
   - Sharpe ratio
   - win rate
   - max drawdown

5. **Validation Checks**
   - sensitivity to delay assumptions
   - robustness across event subsets
   - sanity comparison (e.g. randomized baseline)

---

## Outputs (Persisted in `output/005_backtest/`)

Artifacts produced by this stage:

- `trade_df.pkl`
  - Event-level trade log

- `daily_returns.pkl`
  - Aggregated daily return series

- `equity_curve.pkl`
  - Cumulative return curve

- `performance_summary.pkl`
  - Key performance metrics

- `stress_summary.pkl`
  - Performance across cost and delay scenarios

These outputs are registered in the **pipeline manifest** and form the
basis for final evaluation.

---

## Notes

This stage operates exclusively on artifacts generated in
**004_strategy** and **002_enrich**, ensuring that strategy construction
and validation remain cleanly separated within the pipeline.

All results must be reproducible from persisted artifacts.

---

## Design Principles

- **Deterministic pipeline**  
  Signals and trades are generated only from persisted upstream artifacts.

- **Separation of concerns**  
  Strategy construction (004) is distinct from validation (005).

- **Data integrity**  
  Trade construction must use realized returns (`returns_full`), not
  intermediate analysis structures.

- **Transparent mapping**  
  The transformation from event → trade → PnL must be explicit.

- **Artifact persistence**  
  All intermediate and final outputs are saved for inspection and reuse.

---

## Critical Constraint

005 must validate the **full set of signals produced by 004**, not a
manually reduced subset.


### 1. Imports and Environment Setup
### Provide the necessary imports required for to to proceed.   

In [1]:
import pandas as pd
import numpy as np
import pickle
from datetime import datetime, UTC
import math
from pathlib import Path
import matplotlib.pyplot as plt

### 2. Prepare the environment for the notebook

In [2]:
startdate = "2023-01-01"
trading_days = 252
frequency = "1d"

universe = [
    "BTCUSDT",   # Bitcoin
    "ETHUSDT",   # Ethereum
    "BNBUSDT",   # Binance Coin
    "SOLUSDT",   # Solana
    "XRPUSDT",   # Ripple
    "ADAUSDT",   # Cardano
    "DOGEUSDT",  # Dogecoin
    "AVAXUSDT",  # Avalanche
    "LTCUSDT"    # Litecoin
]

execution_delay = [0, 1, 2, 3]
execution_cost_bps = [20, 30, 40]

stage_label = "005_backtest"

OUTPUT_ROOT = Path("../output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = OUTPUT_ROOT / "manifest.pkl"

DOWNLOAD_DIR = OUTPUT_ROOT / "001_download"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ENRICH_DIR = OUTPUT_ROOT / "002_enrich"
ENRICH_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR = OUTPUT_ROOT / "003_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

STRATEGY_DIR = OUTPUT_ROOT / "004_strategy"
STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

BACKTEST_DIR = OUTPUT_ROOT / "005_backtest"
BACKTEST_DIR.mkdir(parents=True, exist_ok=True)

inspection_window = 20

sigma_threshold = 3

observation_window_length = 10
observation_window = range(1, observation_window_length + 1)

holding_period = 1


### 3. Loading the manifest pickle file

In [3]:
if MANIFEST_FILE.exists():
    manifest = pd.read_pickle(MANIFEST_FILE)
else:
    manifest = {}

manifest.setdefault(stage_label, {})

{}

### 4. Load the pickle files from the previous stage
We use these file contents for our analysis.

In [4]:
required_files = {
    "event_response_expanded": STRATEGY_DIR / "event_response_expanded.pkl",
}

missing_files = [
    name for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(f"Missing required artifacts: {missing_files}")

event_response_expanded = pd.read_pickle(
    required_files["event_response_expanded"]
)

print("Artifacts loaded successfully")

print("\nShape:")
print(event_response_expanded.shape)

print("\nColumns:")
print(event_response_expanded.columns.tolist())

print("\nSample:")
print(event_response_expanded.head())

print("\nNull count:")
print(event_response_expanded.isnull().sum().sum())

Artifacts loaded successfully

Shape:
(2304, 14)

Columns:
['event_date', 'reference_coin', 'target_coin', 'sigma', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10']

Sample:
                 event_date reference_coin target_coin  sigma     lag_1  \
0 2023-08-03 00:00:00+00:00        LTCUSDT     ADAUSDT     -3  0.004098   
1 2023-08-03 00:00:00+00:00        LTCUSDT    AVAXUSDT     -3 -0.003213   
2 2023-08-03 00:00:00+00:00        LTCUSDT     BNBUSDT     -3  0.001663   
3 2023-08-03 00:00:00+00:00        LTCUSDT     BTCUSDT     -3 -0.002569   
4 2023-08-03 00:00:00+00:00        LTCUSDT    DOGEUSDT     -3 -0.001764   

      lag_2     lag_3     lag_4     lag_5     lag_6     lag_7     lag_8  \
0 -0.001020 -0.006129 -0.003768  0.023384  0.011089 -0.014955 -0.009447   
1  0.002417  0.011254 -0.011924  0.021722 -0.004724 -0.011867 -0.004804   
2  0.007884  0.000412 -0.004938  0.014475 -0.006115 -0.010254 -0.005387   
3 -0.001855  0.000888  0.003721  

### 5. Signal Construction

In [8]:
df = event_response_expanded.copy()

required_cols = ["event_date", "reference_coin", "target_coin"]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

lag_cols = sorted(
    [c for c in df.columns if c.startswith("lag_")],
    key=lambda x: int(x.split("_")[1])
)

print("Detected lag columns:", lag_cols)

# we require at least lag_1 ... lag_6
if len(lag_cols) < 6:
    raise ValueError("Not enough lag columns for signal construction")

# Example: lag_5 predicts lag_6
signal_lag = "lag_5"
target_lag = "lag_6"

if signal_lag not in df.columns or target_lag not in df.columns:
    raise ValueError("Required lag columns not present")

signal_df = df[
    ["event_date", "reference_coin", "target_coin", signal_lag, target_lag]
].copy()

signal_df["signal"] = np.sign(signal_df[signal_lag])

signal_df["realized"] = np.sign(signal_df[target_lag])

signal_df["correct_direction"] = (
    signal_df["signal"] == signal_df["realized"]
).astype(int)

signal_df = signal_df[signal_df["signal"] != 0]

print("\nSignal DF shape:", signal_df.shape)

print("\nSignal distribution:")
print(signal_df["signal"].value_counts(normalize=True))

print("\nHit rate:")
print(signal_df["correct_direction"].mean())

Detected lag columns: ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10']

Signal DF shape: (2294, 8)

Signal distribution:
signal
-1.0    0.551439
 1.0    0.448561
Name: proportion, dtype: float64

Hit rate:
0.486050566695728


### 006 — Trade Construction (DIRECT MAPPING)

In [13]:
trade_df = signal_df.copy()

trade_df["position"] = trade_df["signal"]

trade_df["signal_return"] = (
    trade_df["position"] * trade_df["lag_6"]
)

print("trade_df shape:", trade_df.shape)

print("\nColumns:")
print(trade_df.columns.tolist())

print("\nSample:")
print(trade_df.head())

print("\nReturn stats:")
print(trade_df["signal_return"].describe())

print("\n")
print(trade_df[["lag_6", "position", "signal_return"]].head())

trade_df shape: (2294, 10)

Columns:
['event_date', 'reference_coin', 'target_coin', 'lag_5', 'lag_6', 'signal', 'realized', 'correct_direction', 'position', 'signal_return']

Sample:
                 event_date reference_coin target_coin     lag_5     lag_6  \
0 2023-08-03 00:00:00+00:00        LTCUSDT     ADAUSDT  0.023384  0.011089   
1 2023-08-03 00:00:00+00:00        LTCUSDT    AVAXUSDT  0.021722 -0.004724   
2 2023-08-03 00:00:00+00:00        LTCUSDT     BNBUSDT  0.014475 -0.006115   
3 2023-08-03 00:00:00+00:00        LTCUSDT     BTCUSDT  0.019690 -0.006716   
4 2023-08-03 00:00:00+00:00        LTCUSDT    DOGEUSDT  0.020952  0.004797   

   signal  realized  correct_direction  position  signal_return  
0     1.0       1.0                  1       1.0       0.011089  
1     1.0      -1.0                  0       1.0      -0.004724  
2     1.0      -1.0                  0       1.0      -0.006115  
3     1.0      -1.0                  0       1.0      -0.006716  
4     1.0       1

### 007 — Daily Aggregation

In [12]:
daily_returns = (
    trade_df
    .groupby("event_date")["signal_return"]
    .mean()
    .sort_index()
)

print("daily_returns length:", len(daily_returns))

print("\nHead:")
print(daily_returns.head())

print("\nStats:")
print(daily_returns.describe())

daily_returns length: 109

Head:
event_date
2023-08-03 00:00:00+00:00    0.001433
2023-08-15 00:00:00+00:00   -0.004829
2023-08-16 00:00:00+00:00    0.008307
2023-08-17 00:00:00+00:00   -0.015491
2023-09-11 00:00:00+00:00    0.014273
Name: signal_return, dtype: float64

Stats:
count    109.000000
mean       0.000167
std        0.032896
min       -0.126048
25%       -0.015491
50%        0.002158
75%        0.015934
max        0.089183
Name: signal_return, dtype: float64


In [ ]:
### 008 — Equity Curve + Performance

In [15]:

equity_curve = (1 + daily_returns).cumprod()

total_return = equity_curve.iloc[-1] - 1

volatility = daily_returns.std() * np.sqrt(trading_days)

sharpe = (
    daily_returns.mean() /
    daily_returns.std()
) * np.sqrt(trading_days)

drawdown = equity_curve / equity_curve.cummax() - 1
max_drawdown = drawdown.min()

print("\n===== PERFORMANCE =====")
print("total_return:", total_return)
print("volatility:", volatility)
print("sharpe:", sharpe)
print("max_drawdown:", max_drawdown)

print("\nEquity head:")
print(equity_curve.head())

print("\nEquity tail:")
print(equity_curve.tail())


===== PERFORMANCE =====
total_return: -0.04009711862023724
volatility: 0.5222129626341407
sharpe: 0.08051693252321862
max_drawdown: -0.36674481146449966

Equity head:
event_date
2023-08-03 00:00:00+00:00    1.001433
2023-08-15 00:00:00+00:00    0.996597
2023-08-16 00:00:00+00:00    1.004875
2023-08-17 00:00:00+00:00    0.989309
2023-09-11 00:00:00+00:00    1.003429
Name: signal_return, dtype: float64

Equity tail:
event_date
2026-01-25 00:00:00+00:00    1.028680
2026-01-29 00:00:00+00:00    1.065238
2026-01-31 00:00:00+00:00    0.930967
2026-02-04 00:00:00+00:00    0.936404
2026-02-05 00:00:00+00:00    0.959903
Name: signal_return, dtype: float64


### 009 — Cost Model

In [16]:

cost_levels = {
    "20bps": 0.002,
    "30bps": 0.003,
    "40bps": 0.004,
}

results = {}

for label, cost in cost_levels.items():

    df_cost = trade_df.copy()

    df_cost["net_return"] = df_cost["signal_return"] - cost

    daily = (
        df_cost
        .groupby("event_date")["net_return"]
        .mean()
        .sort_index()
    )

    equity = (1 + daily).cumprod()

    total_return = equity.iloc[-1] - 1

    vol = daily.std() * np.sqrt(trading_days)

    sharpe = (
        daily.mean() /
        daily.std()
    ) * np.sqrt(trading_days)

    dd = equity / equity.cummax() - 1
    max_dd = dd.min()

    results[label] = {
        "total_return": total_return,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
    }

for k, v in results.items():
    print(f"\n===== {k} =====")
    for metric, val in v.items():
        print(f"{metric}: {val}")


===== 20bps =====
total_return: -0.22844234933395458
sharpe: -0.8846065631819231
max_drawdown: -0.4160375366424479

===== 30bps =====
total_return: -0.30838011958117495
sharpe: -1.367168311034493
max_drawdown: -0.43926040765290175

===== 40bps =====
total_return: -0.3801040641468302
sharpe: -1.8497300588870633
max_drawdown: -0.4615819664640153


In [17]:
# ============================================
# DEBUG — Verify migration consistency
# ============================================

print("\n=== TRADE DF CHECK ===")
print("rows:", len(trade_df))
print("mean signal_return:", trade_df["signal_return"].mean())
print("std signal_return:", trade_df["signal_return"].std())

print("\n=== DAILY (NO COST) CHECK ===")
daily_check = (
    trade_df
    .groupby("event_date")["signal_return"]
    .mean()
    .sort_index()
)

print("daily mean:", daily_check.mean())
print("daily std:", daily_check.std())

print("\n=== COST APPLICATION CHECK (20bps) ===")
df_cost_check = trade_df.copy()
df_cost_check["net_return"] = df_cost_check["signal_return"] - 0.002

daily_cost_check = (
    df_cost_check
    .groupby("event_date")["net_return"]
    .mean()
    .sort_index()
)

print("daily mean (after cost):", daily_cost_check.mean())
print("daily std (after cost):", daily_cost_check.std())


=== TRADE DF CHECK ===
rows: 2294
mean signal_return: -0.00063946534718364
std signal_return: 0.07047922027547854

=== DAILY (NO COST) CHECK ===
daily mean: 0.0001668531185522349
daily std: 0.03289632453676671

=== COST APPLICATION CHECK (20bps) ===
daily mean (after cost): -0.0018331468814477667
daily std (after cost): 0.03289632453676671
